In [ ]:
subject_id = "170"
# set by the runner (papermill); None means: find settings.toml from cwd
config_path = None
# set by the runner, e.g. derivatives_root -> a test folder
path_overrides = None


In [ ]:
import mne
from mne.preprocessing import ICA
import autoreject
from pathlib import Path
import pandas as pd
from datetime import datetime
import json
import os
import numpy as np
from spectral.ica import compute_ica
from spectral.viz import plot_step, plot_bad_channels, plot_epochs
from spectral.specparam import (
    specparam2pandas,
    plot_aperiodic_topomaps,
    examine_spectra,
    plot_models,
)
from spectral.utils import ProjectPaths,print_timestamp

""" config = read_parameters()
my_paths = config_project(subject)
project_path = my_paths["project_path"]
figures_path = my_paths["figures_path"]
specparam_path = my_paths["specparam_path"]

Path(specparam_path).mkdir(parents=True, exist_ok=True)
 """


paths = ProjectPaths(subject_id, config_path=config_path, path_overrides=path_overrides)
# Create all directories
paths.create_directories()

#paths = ProjectPaths(subject)
# Create all directories
#paths.create_directories()

# This is analysis output, so it goes in the analysis folder
specparam_path = paths.specparam
project_path = paths.root
figures_path = paths.figures
preprocessed_path = paths.preprocessed
epochs_path = paths.epochs
# Print paths to verify
print_timestamp("Setting up project paths")
paths.show()

mne.viz.set_browser_backend("matplotlib")
# mne.viz.set_browser_backend("qt")
mne.set_config("MNE_BROWSER_THEME", "light")
# matplotlib.use("Agg")
#print_date_time()


In [ ]:
report = mne.Report(title=f"Report for subject {subject_id}", subject=subject_id)
#report.add_raw(raw=clean_raw, title="Raw", psd=True, butterfly=True, scalings='auto')

In [ ]:
from specparam.plts.spectra import plot_spectra
from specparam import SpectralGroupModel


fg = SpectralGroupModel(
    peak_width_limits=[1, 6],
    min_peak_height=0.15,
    peak_threshold=2.0,
    max_n_peaks=6,
    verbose=False,
)

freq_range = [2, 35]

In [ ]:
epochs_interpolated = mne.read_epochs(
    f"{paths.analysis}/sub-{subject_id}_interpolated-epo.fif", preload=True
)

In [ ]:
psd = epochs_interpolated.compute_psd().average()
spectra, freqs = psd.get_data(return_freqs=True)
# Initialize a FOOOFGroup object, with desired settings

# Define the frequency range to fit

with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
    fg.fit(freqs, spectra, freq_range)
fg.plot()

In [ ]:
channel_names = epochs_interpolated.info["ch_names"]
df_channels = pd.DataFrame({"ID": range(len(channel_names)), "ch": channel_names})

df = specparam2pandas(fg)
df = df.merge(df_channels, on="ID")
df["sub_id"] = subject_id


# Get the current date and time
now = datetime.now()
df["timestamp"] = now
#df["nr_intepolated_channels"] = len(epochs_ar.info["bads"])
#df["nr_dropped_ica"] = len(ica.exclude)
#df["nr_retained_ica"] = ica.n_components_ - len(ica.exclude)
# Create a new list of column names
cols = ["ch"] + [col for col in df.columns if col != "ch"]

# Reorder the columns
df = df[cols]

# Plot R and exponent across the scalp

In [ ]:
fg.to_df(None)

In [ ]:
import matplotlib.pyplot as plt

# Offset, exponent and fit quality across the scalp.
fig = plot_aperiodic_topomaps(
    fg, epochs_interpolated.info, subject_id,
    params=("offset", "exponent", "r_squared"),
)
plt.show()
report.add_figure(fig, title="Offset, Exponent and R_squared values")


In [ ]:
# Compare the power spectra between low and high exponent channels
fig = examine_spectra(fg, subject_id)
plt.show()
report.add_figure(
    fig, title="Examples of spectra as a function of exponent and R_squared values"
)


In [ ]:
plot_models(fg, param_choice="exponent")


In [ ]:
plot_models(fg, param_choice="r_squared")


In [ ]:
df.to_csv(f"{paths.specparam}/sub-{subject_id}-specparam.csv", index=False)
print(f"Subject {subject_id} done")

In [ ]:
report.save(
    f"{paths.reports}/sub-{subject_id}_report_manual_specparam.html", overwrite=True
)